# Notebook 6 — Counting Steps & Recognising Complexity

**Covers Chapter 3 §3.1, §3.4, §3.5.** The concrete programmer's half of complexity: count steps in real programs, recognise complexity classes by inspecting loop structure.

## What you'll be able to do by the end

- Count steps in any While program by hand using the chapter's rule (assignments + boolean condition checks).
- Look at a program and predict its complexity class without running it (just from loop structure).
- Distinguish best/worst/average case.
- Solve exercises 19, 20, 23, 24, 25.

## What's new in the interpreter

Two new functions:

- `count_steps(prog, state)` — count chapter-style steps for one run.
- `step_growth(prog, state_fn, sizes)` — count steps across many input sizes; return a `{n: step_count}` dict you can plot.

Both follow §3.1.4's definition: count `:=` (assignment) and the boolean check that fires inside `if-tt`/`if-ff`/`while-tt`/`while-ff`. The administrative `;` and `skip-;` rules are **not** counted — they're syntactic bookkeeping in the small-step semantics, not work the program is doing.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace,
    A, B, step,
    count_steps, step_growth,        # NEW in chapter 3
    StepBudgetExceeded,
)
print('imports OK')

## §1. The two ingredients of complexity

The chapter (§3.1) starts with two basic ideas:

1. **A measure of complexity** — what we're counting.
2. **A notion of size** — how big the input is.

Combine them: complexity is a *function* from input-size to step-count.

### Time vs space complexity

- **Time complexity** counts *operations performed*. Not wall-clock time — that depends on hardware. Counting steps gives us a hardware-independent measure that still tracks how slow a program will feel.
- **Space complexity** counts *memory used*. In While, that's effectively the number of variables you need.

There's often a **trade-off** — using more memory can save time. Example: in chapter 1 Example 6 (Fibonacci), we keep a copy of the previous value `z` so we don't have to recompute it. Cutting `z` would force a re-derivation in every iteration, slowing the program down.

We focus on **time complexity** for the rest of the chapter.

### What counts as a step?

From §3.1.4, two kinds of steps:

| Kind | Counts? |
|:--|:-:|
| Assignment `x := a` | ✓ |
| Boolean condition check inside `if` or `while` | ✓ |
| The administrative `;` rule | ✗ |
| The `skip-;` rule | ✗ |

Why exclude the admin rules? They're not the program *doing work* — they're the small-step semantics propagating intermediate results around. Counting them would be like counting the parentheses in your code as separate operations.

Different problems can dictate different counting conventions — for sorting we'd count comparisons, for matrix work we'd count entry accesses. As long as the choice is *representative* of how the algorithm works, the asymptotic class will come out the same.

## §2. The parity programs from §3.1.4

The chapter shows two programs that compute the same thing — `a := 1` if `n` is even, else `a := 0`.

**Program 1** (chapter says `n+2` steps):

```
while ¬(n ≤ 1) do (n := n − 2);
a := 1 − n
```

**Program 2** (chapter says `n+3` steps):

```
y := 0;
while y ≤ n − 1 do (y := y + 2);
if y = n then a := 1 else a := 0
```

The chapter wants us to confirm that even though the programs *look* very different, their step counts grow at the same rate (linear in `n`). Let's count and confirm.

In [ ]:
parity_1 = '''
    while !(n <= 1) do (
        n := n - 2
    );
    a := 1 - n
'''

print('Program 1:  chapter formula = n + 2')
print('-' * 40)
for n in [0, 2, 4, 6, 10, 100]:
    actual = count_steps(parity_1, {'n': n})
    expected = n + 2
    print(f'  n = {n:4}: actual = {actual:4}   formula = {expected:4}   {"✓" if actual == expected else "≠ (rounding)"}')

In [ ]:
parity_2 = '''
    y := 0;
    while y <= n - 1 do (
        y := y + 2
    );
    if y = n then
        a := 1
    else (
        a := 0
    )
'''

print('Program 2:  chapter formula = n + 3')
print('-' * 40)
for n in [0, 2, 4, 6, 10, 100]:
    actual = count_steps(parity_2, {'n': n})
    expected = n + 3
    print(f'  n = {n:4}: actual = {actual:4}   formula = {expected:4}')

**Program 1 matches exactly.** Program 2 is *off by one* from the chapter formula — that's because the chapter explicitly drops the `⌊⌋` floor function in §3.1.4 ("ignoring the ⌊⌋ function"). The formula `n+3` is the asymptotic estimate; the exact integer count is `n+4` for even `n`. **For complexity classes this doesn't matter** — both are linear in `n`, so both are `O(n)`.

This is a recurring theme: when counting steps, exact constants matter much less than the *function form* (linear vs quadratic vs logarithmic).

### What if the input isn't a natural number?

The chapter mentions both programs "fail to terminate if the input fails to be a natural number." Let's check program 1 with `n = -1`:

In [ ]:
# n = -1: !(−1 <= 1) is !(true) = false, so loop body doesn't run.
# Then a := 1 − (−1) = 2. The program *terminates* but produces a meaningless answer.
print(run(parity_1, {'n': -1}))    # → {'a': 2, 'n': -1}
print('steps:', count_steps(parity_1, {'n': -1}))

# So the chapter's claim of non-termination is slightly imprecise — what actually
# happens is *graceful misbehaviour*: the program returns garbage, not infinite
# computation. The intended use-case is n ∈ ℕ where the answer makes sense.

## §3. Best, worst, average case

Two inputs of the *same size* may take *different numbers of steps*. Example: searching for a value `v` in a list of `n` items.

- **Best case** — `v` is the first item: 1 step.
- **Worst case** — `v` is last (or absent): `n` steps.
- **Average case** — assuming `v` is equally likely to be at any position: `n/2` steps.

**For most analysis we focus on the worst case.** Best case is too optimistic (you usually can't rely on it). Average case requires probabilistic assumptions about the input. Worst case gives a guarantee — "no matter what input I get, it'll take *at most* this long" — which is what's useful for safety-critical applications and for designing algorithms.

## §4. Recognising complexity classes by inspection (§3.4)

When you look at a program, you can usually predict its complexity class **just from the structure of its loops** — without counting exact steps.

### The loop heuristic

| Pattern | Complexity |
|:--|:-:|
| No loops, just assignments and ifs | `O(1)` (constant) |
| Single loop bounded by `n`, body is `O(1)` | `O(n)` (linear) |
| Single loop bounded by `n`, body is `O(n)` | `O(n²)` (quadratic) |
| Loop counter halves each iteration (e.g. binary search) | `O(log n)` (logarithmic) |
| Loops counter halves AND something linear runs at each level | `O(n log n)` |
| `k` nested loops, each bounded by `n` | `O(n^k)` |
| Loop counter doubles each iteration | `O(log n)` |
| Loop body itself doubles each iteration | `O(2^n)` (exponential) |

**Key rule:** when loops are nested, **multiply** their iteration counts. When loops run sequentially (one after another), **add** them — but the larger one dominates the asymptotic class anyway.

### The class hierarchy

```
O(1)  ⊊  O(log n)  ⊊  O(n)  ⊊  O(n log n)  ⊊  O(n²)  ⊊  O(n³)  ⊊  …  ⊊  O(2^n)
```

Strict containment — for any `m < m'` we have `O(n^m) ⊊ O(n^m')`. Larger class = worse asymptotic behaviour.

**Tractable** = polynomial complexity (`O(n^m)` for some `m`). **Intractable** = anything growing faster than every polynomial (e.g. `O(2^n)`).

## §5. Worked example — `m div 2` (§3.5)

From the chapter, here's a program that computes `m div 2` (integer division by 2):

```
x := 0;
while 1 ≤ m do (
    m := m − 2;
    x := x + 1
)
```

**Step count by hand:**

- 1 step before the loop (`x := 0`).
- The loop runs `⌊m/2⌋` times. Each iteration: 1 boolean check + 2 assignments = 3 steps.
- 1 final boolean check (false, exits the loop).

Total: `1 + 3⌊m/2⌋ + 1 = 3⌊m/2⌋ + 2`. Linear in `m`. So **`O(n)`**.

Let's verify:

In [ ]:
div2_prog = '''
    x := 0;
    while 1 <= m do (
        m := m - 2;
        x := x + 1
    )
'''

print("m div 2:  formula = 3⌊m/2⌋ + 2  (asymptotically O(n))")
print('-' * 50)
for m in [4, 5, 10, 11, 100, 101, 1000]:
    actual = count_steps(div2_prog, {'m': m})
    expected = 3 * (m // 2) + 2
    print(f'  m = {m:5}: actual = {actual:5}   formula = {expected:5}')

Match for even `m`. For odd `m`, my counter is 3 higher than the chapter's formula — that's the loop running one *extra* iteration past `⌊m/2⌋` because the body still fires when `m=1` (true), reducing `m` to `−1` before the next check exits.

Again: doesn't change the asymptotic class. Both are `O(n)`.

## §6. Worked example — restricted While (§3.5)

Imagine we couldn't use `+`, `−`, `×` directly — we could only add or subtract 1, and only compare to 0. The chapter shows that **multiplication** then needs *nested loops*:

```
x := m; y := n; z := m;
while ¬(y = 0) do (
    while ¬(x = 0) do (
        z := z + 1;
        x := x − 1
    );
    y := y − 1;
    x := m
)
```

This computes `z = m × n` using only increments. **The chapter's analysis** (counting only *assignments*, not boolean checks):

- 3 init steps.
- Outer loop runs `n` times. Each outer iteration: inner loop + 2 final assignments.
- Inner loop runs `m` times, 2 assignments each = `2m`.
- Per outer iteration: `2m + 2`.
- Total: `3 + n(2m + 2) = 3 + 2mn + 2n`.

When `m = n`, this is `2n² + 2n + 3` — **quadratic in `n`**, so `O(n²)`.

Note: this conflicts slightly with the §3.1.4 rule (which counted boolean checks too). My `count_steps` does count the checks — so my numbers will be ~1.5× the chapter's formula, but the asymptotic class is the same.

In [ ]:
mul_prog = '''
    x := m; y := n; z := m;
    while !(y = 0) do (
        while !(x = 0) do (
            z := z + 1;
            x := x - 1
        );
        y := y - 1;
        x := m
    )
'''

print('Multiplication via repeated increment')
print("chapter formula (assignments only): 3 + 2mn + 2n")
print("my counter (also counts boolean checks): roughly 3/2× larger")
print('-' * 50)
for (m, n) in [(2, 3), (5, 5), (10, 10), (20, 20)]:
    actual = count_steps(mul_prog, {'m': m, 'n': n})
    formula = 3 + 2*m*n + 2*n
    final = run(mul_prog, {'m': m, 'n': n})
    print(f'  m={m:3}, n={n:3}: m×n={m*n:5}  computed z={final.get("z", 0):5}  steps={actual:5}  (formula {formula})')

### Empirical growth — does it actually look quadratic?

We can use `step_growth` to confirm. If the program is `O(n²)`, doubling `n` should roughly *quadruple* the step count.

In [ ]:
growth = step_growth(mul_prog, lambda n: {'m': n, 'n': n}, [10, 20, 40, 80, 160])

print(f'{"n":>4} | {"steps":>8} | {"steps / n²":>11} | {"ratio to prev":>14}')
print('-' * 50)
prev_steps = None
for n, steps in growth.items():
    ratio = f'{steps / prev_steps:>5.2f}×' if prev_steps else '   —'
    print(f'{n:>4} | {steps:>8} | {steps / (n*n):>11.4f} | {ratio:>14}')
    prev_steps = steps

# The 'steps / n²' column should converge to a constant as n grows — that
# constant IS the leading coefficient of n². The 'ratio to prev' column
# should hover around 4 (because doubling n quadruples the work for O(n²)).

**Two ways to confirm O(n²)** from the data:

1. **`steps / n²` converges to a constant.** That constant is the leading coefficient. If it kept shrinking → would be O(n). If it kept growing → would be O(n³) or worse.
2. **Doubling `n` quadruples `steps`.** That ratio of 4× is the signature of quadratic. For linear it'd be 2×. For cubic, 8×.

This is the *empirical* version of complexity analysis. Useful when the program is too tangled to analyse on paper.

## §7. Exercises

## Exercise 19 — count steps for two equivalent programs

> Both programs compute the same value (the triangular number `i(i+1)/2`). Count the number of steps each takes.

**Program left:**

```
y := 1; a := 0;
while y ≤ i do (
    a := a + y;
    y := y + 1
)
```

**Program right:**

```
x := i × (i+1); y := 0;
while 0 ≤ x do (
    x := x − 2;
    y := y + 1
)
```

**Analysis (left):** init = 2 assignments. Outer loop runs `i` times (`y` from `1` to `i`). Each iteration: 1 check + 2 assignments. Plus 1 final check.

Total: `2 + (3i + 1) + 1 = 3i + 4`. **Linear, `O(n)`.**

**Analysis (right):** init = 2 assignments. The loop runs while `0 ≤ x`. `x` starts at `i(i+1)` (which is always even and non-negative for `i ≥ 0`), decreases by 2 each iteration. So it runs `T = i(i+1)/2 + 1` times before `x` goes negative.

Per iteration: 1 check + 2 assignments. Plus 1 final check.

Total: `2 + 3T + 1 = (3/2)i² + (3/2)i + 4`. **Quadratic, `O(n²)`.**

**Both compute the same answer, but the left program is asymptotically faster.** Verifying:

In [ ]:
ex19_left = '''
    y := 1; a := 0;
    while y <= i do (
        a := a + y;
        y := y + 1
    )
'''

ex19_right = '''
    x := i * (i + 1); y := 0;
    while 0 <= x do (
        x := x - 2;
        y := y + 1
    )
'''

print(f'{"i":>3} | {"left a":>7} | {"right y":>8} | {"left steps":>11} | {"right steps":>12}')
print('-' * 60)
for i in [3, 5, 10, 20, 50]:
    L = run(ex19_left, {'i': i})
    R = run(ex19_right, {'i': i})
    Ls = count_steps(ex19_left, {'i': i})
    Rs = count_steps(ex19_right, {'i': i})
    print(f'{i:>3} | {L.get("a", 0):>7} | {R.get("y", 0):>8} | {Ls:>11} | {Rs:>12}')

Both programs return the same value (and it equals `i(i+1)/2` — the triangular number). But the left is `O(i)` and the right is `O(i²)`. By `i=50`, the left does ~150 steps; the right does ~3,800. **Choosing the right algorithm beats hardware speed-ups.**

## Exercise 20 — primality test step count (worst case)

> For Example 4 (the primality test), count steps as a function of `n` in the worst case.

Recall the program:

```
y := n − 1;
z := 1;
while 2 ≤ y ∧ z = 1 do (
    r := n;
    while y ≤ r do (
        r := r − y
    );
    if r = 0 then z := 0 else skip;
    y := y − 1
)
```

**Worst case:** `n` is prime — so `z` stays `1` throughout, and the outer loop runs all the way down from `y = n−1` to `y = 2`. That's `n − 2` outer iterations.

**Per outer iteration:**
- 1 outer check (`2 ≤ y ∧ z = 1`)
- 1 assignment `r := n`
- inner loop: `r` decreases by `y` each iteration, so runs `⌊n/y⌋` times. Per inner iter: 1 check + 1 assignment = 2. Plus 1 final check.
- 1 if-check; the else-branch (skip) does no work.
- 1 assignment `y := y − 1`

Per outer iteration: `1 + 1 + (2⌊n/y⌋ + 1) + 1 + 1 = 2⌊n/y⌋ + 5`.

Total (worst case):

$$2 + \sum_{y=2}^{n-1} \left(2 \lfloor n/y \rfloor + 5\right) + 1$$

The sum `Σ ⌊n/y⌋` from `y=1` to `n` is asymptotically `n · H_n ≈ n ln n`. So the worst-case complexity is **`O(n log n)`**.

(The chapter says "there is no need to simplify" — this is the kind of expression they're after.)

In [ ]:
prime_prog = '''
    y := n - 1;
    z := 1;
    while 2 <= y & z = 1 do (
        r := n;
        while y <= r do (
            r := r - y
        );
        if r = 0 then
            z := 0
        else (
            skip
        );
        y := y - 1
    )
'''

# Worst case = primes. Take a few primes and check n log n behaviour.
import math
primes = [11, 23, 47, 97, 199, 397]

print(f'{"n (prime)":>10} | {"steps":>6} | {"steps / (n ln n)":>17}')
print('-' * 45)
for n in primes:
    s = count_steps(prime_prog, {'n': n})
    ratio = s / (n * math.log(n))
    print(f'{n:>10} | {s:>6} | {ratio:>17.4f}')

# The ratio should converge as n grows — confirming O(n log n).

## Exercise 23 — classify functions and order them

> For each function, identify the smallest `O(−)` class it belongs to, then put them in rising order.

| | Function | Smallest class | Reasoning |
|:-:|:--|:-:|:--|
| (a) | `2x + 10000` | `O(n)` | Linear (the constant 10000 is dominated by the linear term). |
| (b) | `log x − 1000` | `O(log n)` | Logarithmic. The `−1000` is asymptotically irrelevant. |
| (c) | `x² − x` | `O(n²)` | Quadratic dominates. |
| (d) | `2x` | `O(n)` | Linear (literally `2 · x`). |
| (e) | `500` | `O(1)` | Constant. |
| (f) | `log x + x²` | `O(n²)` | The `x²` dominates `log x`. |
| (g) | `200x − log x` | `O(n)` | Linear dominates. |

**Rising order** (smallest complexity class first):

$$\underbrace{(e)}_{O(1)} \;<\; \underbrace{(b)}_{O(\log n)} \;<\; \underbrace{(a) \equiv (d) \equiv (g)}_{O(n)} \;<\; \underbrace{(c) \equiv (f)}_{O(n^2)}$$

Functions in the same class are equivalent for asymptotic purposes (`a ∼ d` because `O(a) = O(d) = O(n)`).

## Exercise 24 — nested sum

> Compute `Σᵢ₌₁ⁿ Σⱼ₌₁ⁱ j`. Write a While program. Count steps. Identify the smallest complexity class.

### (a) Program

```
x := 0;
i := 1;
while i ≤ n do (
    j := 1;
    while j ≤ i do (
        x := x + j;
        j := j + 1
    );
    i := i + 1
)
```

### (b) Step count

**Two valid answers**, depending on counting convention:

| Convention | Per outer iter | Total | Source |
|:--|:--|:--|:--|
| **Assignments only** | `2 + 2i` | `2 + Σᵢ(2 + 2i) = n² + 3n + 2` | Official sample solutions (matches §3.5's multiplication example) |
| **Assignments + boolean checks** | `3i + 4` | `2 + Σᵢ(3i + 4) + 1 = (3/2)n² + (11/2)n + 3` | Per §3.1.4's stated rule, which is what `count_steps()` follows |

The chapter is internally inconsistent — §3.1.4 counts both kinds of steps, but Ex 24 (and §3.5's multiplication) count only assignments. **Either answer is justifiable**; flag your convention when answering.

Either way, the asymptotic class is the same.

### (c) Smallest class

Quadratic — **`O(n²)`**. (Both formulas are polynomials of degree 2.)

Note: the *value* this program computes is `Σ T_i = O(n³)`, but the *number of steps* needed to compute it is only `O(n²)`. Computing big numbers doesn't necessarily require many steps.

In [ ]:
ex24_prog = '''
    x := 0;
    i := 1;
    while i <= n do (
        j := 1;
        while j <= i do (
            x := x + j;
            j := j + 1
        );
        i := i + 1
    )
'''

# n=80 needs ~10K steps; bump the budget so step_growth doesn't truncate.
growth = step_growth(ex24_prog, lambda n: {'n': n}, [5, 10, 20, 40, 80], max_steps=100_000)
print(f'{"n":>4} | {"steps":>7} | {"steps / n²":>11} | {"x (value)":>10}')
print('-' * 45)
for n, s in growth.items():
    final = run(ex24_prog, {'n': n}, max_steps=100_000)
    print(f'{n:>4} | {s:>7} | {s / (n*n):>11.4f} | {final.get("x", 0):>10}')
# steps / n² should converge — confirming O(n²).

## Exercise 25 — classify four programs

> For each, state the time complexity with a brief justification, then place them in order from smallest to largest.

### (i) Outer/inner loops with conditional decrement

```
d := 0;
while y ≤ r do (
    s := r;
    c := y;
    while c > 0 do (c := c − 1; s := s − 1);
    if ¬(r = s) then (d := d + 1; r := s) else skip
)
```

**Analysis:** outer iterates while `y ≤ r`. Each outer iteration: inner loop runs `y` times (decrementing `c` from `y` to 0), then `r := s` (which is `r − y`). So `r` decreases by `y` each outer iteration → outer runs ⌊r/y⌋ times.

Per outer iteration: ~`3y + 7` steps (constant + inner). Total: `(3y + 7) · (r/y) = 3r + 7r/y`. With `r` and `y` both as inputs, **dominated by `3r`** — so **`O(n)`** where `n = r`.

(This program is yet another implementation of integer division: `d` ends up being `r div y`. The official sample solution agrees: linear in `r`.)

### (ii) Single linear loop with constant tail

```
z := 1;
while 1 ≤ n do (z := z × 2; n := n − 1);
z := z × (2 × m + 1) − 1
```

**Analysis:** loop runs `n` times, 2 assignments + 1 check each. Plus init (1) + final assignment (1) + final check (1). Total ~`3n + few`. **`O(n)`**.

(Computes `z = 2ⁿ × (2m + 1) − 1`.)

### (iii) Loop with fixed bounds

```
z := 1; r := 0; s := 3;
while r ≤ s + 1 do (
    if r < s then (r := r + 1; s := s − 1; z := r)
    else (z := r − s; s := r − 2)
)
```

**Analysis:** all initial values are *literals* (0, 3). The loop progresses through a fixed number of states before exiting. The loop body executes some bounded number of times, regardless of any input. **`O(1)`** — constant time.

(In this program there's no input variable that affects the loop count. Try running it with different external state — answer doesn't change.)

### (iv) Doubly nested loop

```
x := n; y := n; z := n;
while z > 0 do (
    while x > 0 do (n := n + y; x := x − 1);
    x := y;
    z := z − 1
)
```

**Analysis:** outer runs `n` times (`z` from `n` to `0`). Inner runs `n` times each outer iteration (`x` is reset to `y = n` at the end of each outer body). Total: `n × n = n²` body executions. **`O(n²)`**.

### Order from smallest to largest

$$\text{(iii)}_{O(1)} \;<\; \text{(i)}_{O(n)} \equiv \text{(ii)}_{O(n)} \;<\; \text{(iv)}_{O(n^2)}$$

(The official sample solution lists the same per-program classes but its ordering text reads "(iii) and (iv), followed by (ii) and (i) in rising order" — that appears to be a transcription error, since (iv) is `O(n²)`, the largest, and can't be second in a rising list. The sensible reading, consistent with the per-program analysis, is the ordering above.)

In [ ]:
# Empirical check on (iv) — should look quadratic.
ex25_iv = '''
    x := n; y := n; z := n;
    while 1 <= z do (
        while 1 <= x do (
            n := n + y;
            x := x - 1
        );
        x := y;
        z := z - 1
    )
'''
growth = step_growth(ex25_iv, lambda n: {'n': n}, [5, 10, 20, 40])
print('(iv) growth — expect O(n²):')
for n, s in growth.items():
    print(f'  n={n}: {s} steps    (steps/n² = {s/(n*n):.3f})')

## Summary

- A **step** is an assignment or a boolean condition check. The administrative `;` and `skip-;` rules don't count.
- **Worst-case** is the standard convention — gives a guarantee about the upper bound on running time.
- **Loop heuristic** for recognising classes:
  - No loops → `O(1)`
  - 1 loop bounded by `n` → `O(n)`
  - Nested loops → multiply
  - Counter halves each iter → `O(log n)`
- Programs that compute the *same value* can have very different complexity (Exercise 19 — both linear-vs-quadratic versions compute the triangular number).
- The *value* a program computes can be much bigger than the number of *steps* it takes (Exercise 24 — computes `O(n³)`-sized output in `O(n²)` steps).
- Empirical confirmation: `step_growth` lets you watch the growth rate; a stable `steps / n²` ratio confirms quadratic, etc.

**Next: N7 — the formal mathematics behind big-O, big-Ω, big-Θ.** Why these classes form a strict hierarchy, why `O(log n)` sits between constant and linear, and how the propositions in §3.3 let you simplify expressions like `O(3x² + 2x + 5)` to just `O(n²)`.